# DatensEE Quickstart — Colab / Notebook

Export Earth Engine imagery at scale via Cloud Dataflow, directly from a notebook.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michaelfdewitt/datensee/blob/master/notebooks/datensee_quickstart.ipynb)

## 1. Install DatensEE

In [ ]:
!pip install -q "datensee[validation] @ git+https://github.com/michaelfdewitt/datensee.git#subdirectory=cli"

## 2. Authenticate

In Colab, this triggers the interactive Google auth flow.
Outside Colab, ensure ADC is configured (`gcloud auth application-default login`).

In [ ]:
import datensee
from datensee import notebook

notebook.ensure_auth()

## 3. Define the export

Use the built-in demo (Landsat 9 NDVI over SF Bay Area) or define your own
expression with `ee.serializer.encode(image, for_cloud_api=True)`.

In [ ]:
from datensee.api import _demo_expression, _demo_region

# For custom expressions:
# import ee
# ee.Initialize(project="your-project")
# image = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2023-06-01', '2023-09-01').median().normalizedDifference(['SR_B5', 'SR_B4'])
# expression = ee.serializer.encode(image, for_cloud_api=True)
# region = ee.Geometry.Rectangle([-122.5, 37.75, -122.25, 38.0]).getInfo()

expression = _demo_expression()
region = _demo_region()

PROJECT = "your-gcp-project"  # <-- change this

## 4. Tile the region

In [ ]:
grid = datensee.tile(region, scale=30.0)
print(f"{len(grid.tiles)} tiles at {grid.pixel_size:g} {grid.crs} units/px")

## 5. Review export config

In [ ]:
from datensee.config import PipelineConfig, OutputConfig, RunnerConfig
from datensee.expression import clip_expression

# Build a quick config to review
config = PipelineConfig(
    ee_expression=clip_expression(expression, region),
    gee_project=PROJECT,
    tile_grid=grid,
    output=OutputConfig(output_path="gs://your-bucket/output"),
    runner=RunnerConfig(mode="local"),
)

notebook.display_export_summary(config)

## 6. Run the export

For small regions, use `runner="local"` with a local output directory.
For large regions, use `runner="dataflow"` with a GCS output path.

In [ ]:
result = datensee.export(
    ee_expression=expression,
    region=region,
    project=PROJECT,
    output="./datensee-output",
    runner="local",
)

print(f"Tiles OK: {result.tiles_ok}")
print(f"Duration: {result.duration_seconds:.1f}s")

## 7. Monitor a Dataflow job (optional)

If you used `runner="dataflow"`, poll the job with an HTML status display.

In [ ]:
# Uncomment to monitor a Dataflow job:
# final_state = notebook.display_job_progress(
#     result.job_id,
#     project=PROJECT,
#     region="us-central1",
# )

## 8. View the output

Output files are plain Cloud Optimized GeoTIFFs — open them directly in QGIS or any
other GIS tool. To peek at one from Python (needs `rasterio` and `matplotlib`):

```python
import rasterio, matplotlib.pyplot as plt
with rasterio.open("./datensee-output/tile_r0000_c0000.tif") as src:
    plt.imshow(src.read(1), cmap="viridis")
```

## 9. Validate output (optional)

In [ ]:
# from datensee.pixel.validation import validate_output
# report = validate_output("./datensee-output", result.config)
# print(report.render())